# ☄️ 태양계 천체의 역학적 분류 — Google Colab 실습

실제 태양계 천체 카탈로그와 궤도요소를 이용해 **행성·왜소행성·소행성·혜성의 역학적 특성과 분류 기준**을 탐구합니다.

**Colab판 특징**
- PC에 Python/Jupyter를 설치할 필요가 없습니다.
- 공식 GitHub 저장소에서 분석 함수와 데이터를 자동으로 내려받습니다.
- 원본 Jupyter의 핵심인 데이터 탐색, 분류 기준 분석, 2D/3D 궤도 시각화를 유지합니다.
- 무거운 Dash 서버 대신 **Colab 입력폼 + Plotly 대화형 그래프**를 사용합니다.

> 위에서부터 순서대로 실행하세요. 궤도 계산이 느리면 `nps`를 101~201로 낮추면 됩니다.

관련 연구: **조훈 · 손정주, 「태양계 천체의 역학적 분류를 위한 데이터 사이언스 기반 천문 교육 콘텐츠 개발」**


## 0. 실행환경 준비

In [ ]:
# Colab 실행에 필요한 패키지 설치
%pip -q install "plotly==5.15.0" astropy scikit-learn
print("✅ 패키지 설치 완료")


In [ ]:
# 공식 GitHub에서 분석 함수와 전체 카탈로그 자동 복원
from pathlib import Path
from urllib.request import urlretrieve
import base64, zlib, os

BASE = "https://raw.githubusercontent.com/GodTANKS/Solar-System-Dynamical-Classification-Education/main"
WORK = Path("/content/solar_system_dynamics")
WORK.mkdir(parents=True, exist_ok=True)

# 분석 함수
helper = WORK / "solarsystem_analysis13.py"
urlretrieve(f"{BASE}/src/solarsystem_analysis13.py", helper)

# 전체 CSV(3,066개 소천체)를 압축 Base64 조각 12개에서 정확히 복원
encoded_parts = []
for i in range(1, 13):
    part = WORK / f"catalog-{i:02d}.txt"
    urlretrieve(f"{BASE}/data/catalog_b64/part-{i:02d}.txt", part)
    encoded_parts.append(part.read_text(encoding="ascii").strip())

csv_bytes = zlib.decompress(base64.b64decode("".join(encoded_parts)))
csv_path = WORK / "dwarfplanet_asteroid_comet_analysis.csv"
csv_path.write_bytes(csv_bytes)

os.chdir(WORK)
print("✅ 작업 폴더:", WORK)
print("✅ 전체 카탈로그 복원:", csv_path.stat().st_size, "bytes")


In [ ]:
# 분석 라이브러리와 연구용 함수 불러오기
from solarsystem_analysis13 import *
from astropy.time import Time
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

print("✅ 라이브러리 준비 완료")


## 1. 데이터 수집

In [ ]:
# 현재 시각의 8개 행성 궤도요소 + 소천체 카탈로그 결합
now_date = Time.now()
df_p = planet_orbit_element(now_date.jd)
df_n = pd.read_csv("dwarfplanet_asteroid_comet_analysis.csv", dtype={"pdes": str})

df_p["pdes"] = df_p["pdes"].astype(str)
df = pd.concat([df_p, df_n], ignore_index=True)

print("분석 기준 UTC:", now_date.iso)
print("행성:", len(df_p), "개 / 소천체:", len(df_n), "개 / 전체:", len(df), "개")
display(df.head(12))


## 2. 데이터 탐색

In [ ]:
print("[주요유형 개수]")
display(df["pr_class"].value_counts().to_frame("count"))
print("[세부유형 개수]")
display(df["dt_class"].value_counts().to_frame("count"))
print("[주요 수치형 속성 기술통계]")
display(df[["q","ad","a","e","i","t_jup","moid","per_y"]].describe())


## 3. 데이터 처리 — 천체 유형별 데이터 세트

In [ ]:
# 원본 연구의 천체 유형별 데이터 세트 구성
sets = {
    "df": df,
    "df_p": df[df.pr_class == "Planet"],
    "df_d": df[df.pr_class == "Dwarfplanet"],
    "df_a": df[df.pr_class == "Asteroid"],
    "df_c": df[df.pr_class == "Comet"],
}
sets.update({
    "df_p_1": sets["df_p"][sets["df_p"].dt_class == "TP"],
    "df_p_2": sets["df_p"][sets["df_p"].dt_class == "JP"],
    "df_a_0": sets["df_a"][sets["df_a"].pha == "Y"],
    "df_a_1": sets["df_a"][sets["df_a"].neo == "Y"],
    "df_a_2": sets["df_a"][sets["df_a"].dt_class == "MBA"],
    "df_a_3": sets["df_a"][sets["df_a"].dt_class == "TJN"],
    "df_a_4": sets["df_a"][sets["df_a"].dt_class == "TNO"],
    "df_c_1": sets["df_c"][sets["df_c"].neo == "Y"],
    "df_c_2": sets["df_c"][sets["df_c"].dt_class.isin(["Short-period_JFC","Short-period_HTC"])],
    "df_c_3": sets["df_c"][sets["df_c"].dt_class == "Short-period_JFC"],
    "df_c_4": sets["df_c"][sets["df_c"].dt_class == "Short-period_HTC"],
    "df_c_5": sets["df_c"][sets["df_c"].dt_class == "Long-period"],
})

def dataframe_input(name):
    return sets[name].copy()

summary = pd.DataFrame({k: [len(v)] for k, v in sets.items()}, index=["천체 수"]).T
summary.columns = ["count"]
display(summary)


## 4. 천체 유형별 역학적 분류 기준 찾기

In [ ]:
# 목성 티세랑(Tj=2, 3) 경계선 생성
def jupiter_tisserand():
    ja = float(df_p.loc[df_p["full_name"] == "Jupiter", "a"].iloc[0])
    aa = np.linspace(0.1, 40, 400)
    frames = []
    for tj in [2, 3]:
        val = 1 - 0.25 * ((tj - ja/aa)**2) * (ja/aa)
        mask = val >= 0
        frames.append(pd.DataFrame({"name": f"Tj{tj}", "a": aa[mask], "e": np.sqrt(val[mask])}))
    return pd.concat(frames, ignore_index=True)

df_tj = jupiter_tisserand()


In [ ]:
#@title 산점도로 분류 기준 탐색
dataset_name = "df" #@param ["df","df_p","df_d","df_a","df_a_0","df_a_1","df_a_2","df_a_3","df_a_4","df_c","df_c_1","df_c_2","df_c_3","df_c_4","df_c_5"]
x_col = "a" #@param ["q","ad","a","e","i","om","w","t_jup","moid","per","per_y"]
y_col = "e" #@param ["q","ad","a","e","i","om","w","t_jup","moid","per","per_y"]
color_col = "dt_class" #@param ["pr_class","dt_class","neo","pha"]
show_tj = True #@param {type:"boolean"}

d = dataframe_input(dataset_name)
fig = px.scatter(d, x=x_col, y=y_col, color=color_col, symbol=color_col,
                 hover_name="full_name",
                 hover_data=["pdes","q","ad","a","e","i","t_jup","moid","pr_class","dt_class","neo","pha"],
                 title=f"{dataset_name}: {x_col} - {y_col}")
if show_tj and x_col == "a" and y_col == "e":
    fig.add_traces(list(px.line(df_tj, x="a", y="e", color="name", line_dash="name").select_traces()))
fig.update_layout(height=650)
fig.show()


In [ ]:
#@title 상자수염 그래프로 유형별 분포 비교
box_dataset = "df" #@param ["df","df_p","df_d","df_a","df_a_0","df_a_1","df_a_2","df_a_3","df_a_4","df_c","df_c_1","df_c_2","df_c_3","df_c_4","df_c_5"]
box_x = "dt_class" #@param ["pr_class","dt_class","neo","pha"]
box_y = "a" #@param ["q","ad","a","e","i","om","w","t_jup","moid","per","per_y"]

d = dataframe_input(box_dataset)
fig = px.box(d, x=box_x, y=box_y, color=box_x, points="outliers", hover_name="full_name")
fig.update_traces(boxmean=True)
fig.update_layout(height=600, showlegend=False)
fig.show()


## 5. 개별 천체의 2D·3D 궤도 분석

In [ ]:
# 원본 Notebook의 궤도 분석 흐름을 Colab용으로 묶은 함수
def individual_to_dat_df(object_pdes_list, start_time, finish_time, time_hour):
    s_jd = Time(start_time, format='iso', scale='utc').jd
    f_jd = Time(finish_time, format='iso', scale='utc').jd
    t_d = time_hour / 24
    frames_time = [planet_datetimeframe(s_jd, f_jd, t_d)]
    frames_basic = [df_p]
    for pdes in object_pdes_list:
        obj = df[df['pdes'].astype(str) == str(pdes)]
        obj = obj[obj.pr_class != 'Satellite']
        if obj.empty:
            print(f"⚠️ pdes={pdes}를 찾지 못했습니다.")
            continue
        frames_time.append(object_datetimeframe(obj, s_jd, f_jd, t_d))
        frames_basic.append(obj)
    dat = pd.concat(frames_time, ignore_index=True)
    basic = pd.concat(frames_basic, ignore_index=True)
    labeled = pr_dt_class_labelencoder(dat)
    plot_dat = labeled.drop(columns=['pr_class','dt_class','neo','pha','pr_label','dt_label'], errors='ignore')
    plot_dat['code'] = np.where(labeled['pr_class'].eq('Planet'), 2, 1)
    return plot_dat, basic

def orbit_figures(plot_dat, basic, nps=201, text_names=True):
    text_col = 'full_name' if text_names else None
    fig3d = px.scatter_3d(plot_dat, x='x', y='y', z='z', animation_frame='cal(TDB)',
                          size='code', size_max=9, hover_name='full_name', text=text_col)
    fig3d.add_trace(go.Scatter3d(x=[0], y=[0], z=[0], mode='markers', marker=dict(color='orange', size=5), name='Sun'))
    for _, row in basic.iterrows():
        _, x, y, z = orbit_line(row.a, row.e, row.i, row.om, row.w, nps)
        fig3d.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='lines', name=row.full_name, line=dict(width=3)))
    fig3d.update_layout(height=720, scene=dict(aspectmode='cube'))

    line_df = df_orbit_line_xyz(basic, nps)
    fig2d = px.line(line_df, x='x', y='y', color='full_name', hover_name='full_name')
    fig2d.add_trace(go.Scatter(x=[0], y=[0], mode='markers', marker=dict(color='orange', size=9), name='Sun'))
    fig2d.update_yaxes(scaleanchor='x', scaleratio=1)
    fig2d.update_layout(height=650)
    return fig3d, fig2d


In [ ]:
#@title 분석할 천체와 시간 설정
pdes_text = "1" #@param {type:"string"}
start_time = "2023-06-01 12:00:00" #@param {type:"string"}
finish_time = "2023-06-02 12:00:00" #@param {type:"string"}
time_interval_hour = 24.0 #@param {type:"number"}
nps = 201 #@param {type:"slider", min:101, max:801, step:50}
show_names = True #@param {type:"boolean"}

pdes_list = [x.strip() for x in pdes_text.split(',') if x.strip()]
plot_dat, basic = individual_to_dat_df(pdes_list, start_time, finish_time, time_interval_hour)
print("분석 천체:", basic[['full_name','pdes','pr_class','dt_class']].tail(max(1, len(pdes_list))).to_dict('records'))
fig3d, fig2d = orbit_figures(plot_dat, basic, nps=nps, text_names=show_names)
fig3d.show()
fig2d.show()


## 6. 분류 정답 확인과 해석

In [ ]:
#@title 천체의 기존 분류 확인
answer_pdes = "1" #@param {type:"string"}
ids = [x.strip() for x in answer_pdes.split(',') if x.strip()]
answer = df[df['pdes'].astype(str).isin(ids)].copy()
display(answer[['full_name','pdes','pr_class','dt_class','neo','pha','a','e','i','t_jup','moid']])


### 탐구 질문
1. 행성·왜소행성·소행성·혜성은 장반경, 이심률, 경사각에서 어떤 차이를 보이나요?
2. 근지구천체(NEO)와 지구위협천체(PHA)는 `moid`와 궤도 특성에서 어떤 경향을 보이나요?
3. 혜성 분류에서 목성 티세랑(`t_jup`)이 어떤 역할을 하나요?
4. 같은 주요유형 안에서도 세부유형을 구분할 수 있는 속성 조합은 무엇인가요?
5. 한 천체의 3D 궤도 모양과 표의 궤도요소를 함께 보았을 때 어떤 분류 근거를 설명할 수 있나요?

---
**공식 코드·데이터 저장소:** https://github.com/GodTANKS/Solar-System-Dynamical-Classification-Education  
**통합 천문학 실습 홈페이지:** https://GodTANKS.github.io/astronomy-data-science/  
**논문 모음:** https://GodTANKS.github.io/astronomy-data-science/papers/
